# Datamine LIST Process Example

This notebook demonstrates how to configure and run the **`list`** process wrapper in `dmstudio`.

## Process Description

## LIST

Display any database file in a standard format.

The Data Definition is first displayed, followed by each record. If the output line is longer than 80 characters, it is wrapped around.

Each numeric field occupies 12 characters. The format of the field changes with the magnitude of the number, to preserve as much accuracy as possible. Each alphanumeric field occupies as many characters as required, but is preceded by 4 blanks to separate it from the previous field. At the end of the Data Definition and of each page of output, the message:-

## >>> PRESS <RETURN> TO CONTINUE (OR ! TO TERMINATE) >

is displayed. If the response is !<return> then the **LIST** process is terminated. This allows part of the file to be displayed. This message does not appear if **LIST** is invoked from a macro.

The page length is controlled by the @**PROMPT** parameter. The default is 20 records; this may be altered to, say, 60 records by setting @**PROMPT** =60. @**PROMPT** = 0 removes all prompting, except following the Data Definition. Each page of data is headed by the field names.

If an index file is being displayed, then first the Data Definition of the index will be displayed, followed by all data records to which the index refers.

If the input is a catalogue file (as created by the **[LISTDR](<listdr.md>)** process) then all files in the catalogue are listed in turn.

**Note** : The alias L is the alias for the [LISTC](<listc.md>) command; however, if the file should contain any numeric fields, then **LISTC** branches automatically to the LIST process.

### Input Files:

* **in** (Table):
  File to be displayed. If IN is a catalogue file, then all the files in the catalogue
  will be displayed.
  Required=Yes

* **fieldlst** (Undefined):
  File to supply selected fields.
  Required=No

### Output Files:

### Fields:

* **fields** (Undefined : Undefined):
  Optional first listed field. None specified means all.
  Default=Undefined
  Required=No

* **fieldnam** (Character : FIELDLST):
  Field in FIELDLST to identify selected fields.
  Default=Undefined
  Required=No

### Parameters:

* **prompt**:
  Page length for display; 0=infinite (20).
  Range=Undefined
  Values=Undefined
  Default=20
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('list')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute list
print("Running list...")
dm_cmd.list(
    in_i='t_assays',  # required
    # fieldlst_i='optional',  # optional
    # fields_f=['AU'],  # optional
    # fieldnam_f='optional',  # optional
    # prompt_p=20,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("list execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_list_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")